<a href="https://colab.research.google.com/github/ShaunGves/FlyRank-AI/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShaunGves/FlyRank-AI/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/ShaunGves/FlyRank-AI.git
%cd FlyRank-AI

from google.colab import userdata
import os
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("Token loaded successfully" if os.environ["HF_TOKEN"] else "Token missing")

import duckdb
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("""
CREATE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN '{}'
);
""".format(os.environ["HF_TOKEN"]))

Cloning into 'FlyRank-AI'...
remote: Enumerating objects: 146, done.
remote: Counting objects: 100% (146/146), done.
remote: Compressing objects: 100% (102/102), done.
remote: Total 146 (delta 52), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (146/146), 1.89 MiB | 11.44 MiB/s, done.
Resolving deltas: 100% (52/52), done.
/content/FlyRank-AI
Token loaded successfully


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My rule flags pages as "AI opportunities" using two signals. First, search volume (gsc_impressions) — the same signal behind FlyRank's "quick-win" logic, since a page needs enough visibility to matter. Second, engagement (ga4_engaged_sessions) — pages with real engagement are more likely to have the depth/quality AI tools might reference. My rule: flag a page as an opportunity if it has high impressions AND high engagement, giving it reason code visible_engaged_page, and action label review_for_ai_structure.

In [4]:
import pandas as pd

query_data = """
SELECT content_hash_id, gsc_impressions, ga4_engaged_sessions, sessions_ai
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
"""
df = con.execute(query_data).df().dropna()
df["label"] = (df["sessions_ai"] > 0).astype(int)

# Signal 1: impressions bucketed (using rank to avoid duplicate-edge errors)
df["impr_bucket"] = pd.qcut(df["gsc_impressions"].rank(method="first"), q=3, labels=["low", "medium", "high"])
signal1_table = df.groupby("impr_bucket")["label"].agg(["mean", "count"])
print("Signal 1: Impressions vs AI-session rate")
print(signal1_table)

# Signal 2: engagement bucketed
df["engage_bucket"] = pd.qcut(df["ga4_engaged_sessions"].rank(method="first"), q=3, labels=["low", "medium", "high"])
signal2_table = df.groupby("engage_bucket")["label"].agg(["mean", "count"])
print("\nSignal 2: Engagement vs AI-session rate")
print(signal2_table)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

/tmp/ipykernel_1707/607859847.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  signal1_table = df.groupby("impr_bucket")["label"].agg(["mean", "count"])


Signal 1: Impressions vs AI-session rate
                 mean    count
impr_bucket                   
low          0.000178  2274212
medium       0.000110  2274212
high         0.002145  2274213

Signal 2: Engagement vs AI-session rate
                   mean    count
engage_bucket                   
low            0.000769  2274212
medium         0.000608  2274212
high           0.001056  2274213


/tmp/ipykernel_1707/607859847.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  signal2_table = df.groupby("engage_bucket")["label"].agg(["mean", "count"])


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

I combine two signals into one score — percentile rank of gsc_impressions (50% weight) and percentile rank of ga4_engaged_sessions (50% weight). Pages scoring in the top third on both dimensions get reason code visible_engaged_page and action label review_for_ai_structure. Pages strong on only one dimension get a partial reason code (high_visibility_low_engagement or high_engagement_low_visibility) and are set to monitor. This mirrors how FlyRank's own flags combine multiple honest signals into a single actionable score with a clear reason.

In [5]:
import os

# Score: combine impressions rank + engagement rank (both signals from Section 1)
df["impr_rank"] = df["gsc_impressions"].rank(pct=True)
df["engage_rank"] = df["ga4_engaged_sessions"].rank(pct=True)

df["opportunity_score"] = (0.5 * df["impr_rank"]) + (0.5 * df["engage_rank"])

# Reason code + action label
def reason_code(row):
    if row["impr_rank"] > 0.66 and row["engage_rank"] > 0.66:
        return "visible_engaged_page"
    elif row["impr_rank"] > 0.66:
        return "high_visibility_low_engagement"
    elif row["engage_rank"] > 0.66:
        return "high_engagement_low_visibility"
    else:
        return "low_priority"

df["reason_code"] = df.apply(reason_code, axis=1)
df["action_label"] = df["reason_code"].apply(
    lambda x: "review_for_ai_structure" if x == "visible_engaged_page" else "monitor"
)

# Build ranked queue
ranked_queue = df.sort_values("opportunity_score", ascending=False)[
    ["content_hash_id", "gsc_impressions", "ga4_engaged_sessions",
     "opportunity_score", "reason_code", "action_label"]
]

# Write CSV
os.makedirs("work/outputs", exist_ok=True)
ranked_queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

print("Rows written:", len(ranked_queue))
ranked_queue.head(10)


Rows written: 6822637


,content_hash_id,gsc_impressions,ga4_engaged_sessions,opportunity_score,reason_code,action_label
8054586,content_eadb33b5df496f4a,39305,15,1.000000,visible_engaged_page,review_for_ai_structure
7406798,content_eadb33b5df496f4a,33571,12,0.999999,visible_engaged_page,review_for_ai_structure
9767993,content_eadb33b5df496f4a,34606,10,0.999999,visible_engaged_page,review_for_ai_structure
8860963,content_66288edeb93b7c4f,24577,21,0.999999,visible_engaged_page,review_for_ai_structure
7050379,content_eadb33b5df496f4a,32462,10,0.999998,visible_engaged_page,review_for_ai_structure
6867135,content_eadb33b5df496f4a,32665,8,0.999998,visible_engaged_page,review_for_ai_structure
9179913,content_eadb33b5df496f4a,38436,7,0.999998,visible_engaged_page,review_for_ai_structure
7790555,content_eadb33b5df496f4a,31713,8,0.999998,visible_engaged_page,review_for_ai_structure
8131920,content_eadb33b5df496f4a,31133,7,0.999997,visible_engaged_page,review_for_ai_structure
3575129,content_eadb33b5df496f4a,16902,9,0.999996,visible_engaged_page,review_for_ai_structure


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Rows 0–2, 4–11, 13, 15, 17–19 (content_eadb33b5df496f4a, appears 16/20 times): Action: review_for_ai_structure. Why: consistently high impressions (12k–39k) and engaged sessions (6–15) across multiple days in March. What would make it wrong: since this is the same page repeated across many days, my ranked queue is really recommending one page 16 times, not 16 different opportunities — this inflates its apparent priority. A cleaner queue should aggregate to one row per page (e.g., average or max across the month) before ranking.

Row 3 (content_66288edeb93b7c4f, appears twice): Action: review_for_ai_structure. Why: highest engaged sessions in the set (21), paired with strong impressions (24,577). What would make it wrong: if engagement is concentrated in one unusually active day rather than sustained interest.

Row 12 (content_0e03de7680314cd5): Action: review_for_ai_structure. Why: solid impressions (24,335) and engagement (6) place it in the top tier. What would make it wrong: if this page's audience isn't a good fit for AI-referenceable content (e.g., a login page or form).

Row 14 (content_66288edeb93b7c4f, second appearance): Same page as Row 3 — same caveat about day-level duplication applies.

Row 16 (content_ec2e0346994fb5a5): Action: review_for_ai_structure. Why: solid impressions (16,059) with matching engagement (7). What would make it wrong: if this page overlaps heavily in topic with a higher-ranked page, making it a redundant recommendation (consolidation risk, not a true separate opportunity).

Key overall observation: This top-20 list is dominated by just 3 unique pages repeated across days. This is a real weakness in the current rule — it ranks by day-level score without deduplicating by content page first.

In [6]:
top20 = ranked_queue.head(20).reset_index(drop=True)
top20

,content_hash_id,gsc_impressions,ga4_engaged_sessions,opportunity_score,reason_code,action_label
0,content_eadb33b5df496f4a,39305,15,1.000000,visible_engaged_page,review_for_ai_structure
1,content_eadb33b5df496f4a,33571,12,0.999999,visible_engaged_page,review_for_ai_structure
2,content_eadb33b5df496f4a,34606,10,0.999999,visible_engaged_page,review_for_ai_structure
3,content_66288edeb93b7c4f,24577,21,0.999999,visible_engaged_page,review_for_ai_structure
4,content_eadb33b5df496f4a,32462,10,0.999998,visible_engaged_page,review_for_ai_structure
5,content_eadb33b5df496f4a,32665,8,0.999998,visible_engaged_page,review_for_ai_structure
6,content_eadb33b5df496f4a,38436,7,0.999998,visible_engaged_page,review_for_ai_structure
7,content_eadb33b5df496f4a,31713,8,0.999998,visible_engaged_page,review_for_ai_structure
8,content_eadb33b5df496f4a,31133,7,0.999997,visible_engaged_page,review_for_ai_structure
9,content_eadb33b5df496f4a,16902,9,0.999996,visible_engaged_page,review_for_ai_structure


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: The clearest weakness in this top-20 list is duplication by content page rather than by unique opportunity — content_eadb33b5df496f4a alone accounts for 16 of the 20 rows, since the data is at a content+day grain and this page performed consistently well across many days in March. This makes the queue look like it has 20 independent recommendations when it really only surfaces 3 unique pages. A stronger version of this rule would aggregate to one row per content_hash_id (e.g., taking the mean or max score across the month) before ranking, so the top-20 reflects genuine breadth of opportunities rather than one page's daily repeats.

Leakage check: I confirm no product-computed flags (health_score, priority_score, action_type) were used anywhere in this scoring rule — only raw observed signals (gsc_impressions, ga4_engaged_sessions). I also confirm no future-window data was used: this queue is built entirely from month=2026-03, and the label (sessions_ai) used in Section 1's signal checks was only used to validate the signals, never as an input to the score itself in Section 2. The score in Section 2 uses only gsc_impressions and ga4_engaged_sessions — neither is derived from or overlaps with the AI-session label.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ Confirm ] Every section above is filled — markdown thinking AND the code that backs it
- [ Confirm ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ Confirm ] No client names, URLs, or private queries anywhere
- [ Confirm ] My claims use careful words: observed, measured, directional, decision-support
- [ Confirm ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.